# # User Model Definition (PyTorch)

In [1]:
# `load_model` runs once at server start; `infer` runs per request.
# Imports used inside these methods must live inside them (the class
# is cloudpickled when shipped).
#   load_model(context)  — context["worker_index"], context["num_workers"]
#   infer(**kwargs)      — kwargs come from the request JSON body; the
#                          return value must be JSON-serializable.
from nubison_model import NubisonModel, ModelContext


class UserModel(NubisonModel):
    def load_model(self, context: ModelContext) -> None:
        import pickle
        # `src.iris_net.IrisNet` must be importable for unpickle to resolve
        # the class — that's why train_pytorch.ipynb defines it in src/.
        with open("src/weights.pkl", "rb") as f:
            self.net = pickle.load(f)
        self.net.eval()

    def infer(self, sepal_length: float, sepal_width: float,
              petal_length: float, petal_width: float) -> dict:
        import torch
        x = torch.tensor(
            [[sepal_length, sepal_width, petal_length, petal_width]],
            dtype=torch.float32,
        )
        with torch.no_grad():
            logits = self.net(x)
        return {"target": int(logits.argmax(dim=1).item())}


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


# # Model Register

In [ ]:
# Package this class + `src/` (including weights.pkl from train.ipynb)
# as a model in the MLflow Registry. Returns a `models:/<name>/<version>` URI.
#   first arg     — NubisonModel implementation
#   artifact_dirs — comma-separated dirs to ship with the model
#   tags          — model-version tags. mlplatform reads
#                   `inference_server_image` to choose the serving image.
from nubison_model import register

model_id = register(
    UserModel(),
    artifact_dirs="src",
    tags={"inference_server_image": "ghcr.io/nubison/nubison-model:0.0.6"},
)
print(f"Model registered: {model_id}")


# # Model Testing

In [3]:
# Run the model locally over HTTP for a smoke test.
#   test_client(model_id) yields a TestClient — use .post("/infer", json=...)
from nubison_model.Service import test_client

with test_client(model_id) as client:
    result = client.post(
        "/infer",
        json={
            "sepal_length": 5.1, "sepal_width": 3.5,
            "petal_length": 1.4, "petal_width": 0.2,
        },
    )
    print("predicted target:", result.json()["target"])


2026/05/18 11:34:50 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - nubison-model (current: 0.0.10.dev0+ecc5688.20260512100813, required: nubison-model==0.0.10)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


predicted target:

0